In [ ]:
def print_stats(data: list[dict]) -> None:
    """
    Accepts a dataset stored as a list of dictionaries.
    Prints each column's name, and its most common value (for text columns)
    or it's average value (for numeric columns)
    """
    
    # assuming that all dictionaries in the list will have same keys
    columns = data[0].keys()

    for column in columns:
        # determine if this is a text or numeric column
        if type(data[0][column]) == float or type(data[0][column]) == int:
            # process this as a numeric column
            total = 0
            count = 0
            for row in data:
                total += row[column]
                count += 1


                

            average = total / count if count > 0 else 0
            
            print(f"average value for {column}: {average}")
        else:  
            
            # build a dict that counts number of times we've seen each value
            # within this column
            value_counts = dict()
            for row in data:
                value_counts[row[column]] = value_counts.get(row[column], 0) + 1

            # identify the key correponding to the max value
            biggest_count = 0
            most_common = None
            for value, count in value_counts.items():
                if count > biggest_count:
                    biggest_count = count
                    most_common = value
            
            print(f"most common value for {column}: {most_common}")


In [ ]:
def load_from_html(filename: str) -> list[dict]:
    """
    reads a dataset in HTML format. converts numeric data to float.
    raises an AttributeError if the table rows do not all have the same number of values.
    :param filename: the file path of the html data to read
    :return: the dataset as a list of dictionaries - one dict per object in the file
            each dictionary should map column names to values
    """    
    with open(filename, 'r') as file:
        contents = file.read()
        all_rows = []

        head, body = contents.split('</thead>')

        # process the head first, pull out the column names
        head_parts = head.split('<td>')

        columns = []
        for column_name in head_parts[1:]:
            columns.append(
                column_name.replace('</td>', '').replace('</tr>', '').strip()
            )
        
        # strip off some unecessary tags
        body = body.replace('</tr>', '')
        body = body.replace('</tbody>\n</table>', '')

        # process the rest of the text: the table body
        rows_text = body.split('<tr>')
        for row_text in rows_text[1:]: # skip the first, which just has a <tbody> tag
            row_text = row_text.replace('</td>', '')
            values = row_text.split('<td>')
            values = values[1:]

            # check the row has the right number of values in it
            if len(values) != len(columns):
                raise AttributeError(f'wrong number of values in row: {row_text}')

            this_row_dict = dict()
            for i in range(len(columns)):
                this_column = columns[i]
                this_value = values[i].strip()

                # convert to float if the value is a number
                try:
                    this_value = float(this_value)
                except Exception:
                    pass

                this_row_dict[this_column] = this_value
            
            all_rows.append(this_row_dict)
    
    return all_rows